# Multi Model

### AI usage

To better show what code was generated by AI and which was created by myself, I've included 3 icons to each code block to help with this:
* (*) A star means that I used generative AI for the method and no modifications where made by myself.
* (#) A hash means that I used generative AI for the some part of the method however have modified the code to work better.
* (+) A plus means that no generative AI was used for the code.

The primary use was to help with any errors while programming, or when I know what I need a function to perform but not how to write it such as the percision and reacall method.

Google DeepMind. (2024). Gemini 3.6 flash [Large language model]. https://deepmind.google/gemini

In [ ]:
# AI tag - +
verbose = True
export = True
eval_bool = False
use_pretrained = True
Aug = False
pretrained_model = "Models/CSD_no_aug_2_44100_0_2865.pth"
#                  TIMIT,  NUS-48,  Buckeye,   CSD
DATASETS_TO_USE = [False,   False,    False,  True]
save_name = "CSD_aug_2_16000"

In [ ]:
# IMPORTS
# AI tag - +
import os
import csv
import glob
import json
import torch
import random
import librosa
import collections
import numpy as np
import torch.nn as nn
from pathlib import Path
import torch.optim as optim
from json import JSONEncoder
from tqdm.notebook import tqdm
import torch.nn.functional as F
from torch.utils.data import Dataset
import torchaudio.functional as audio_F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader, random_split

## Alphabet Dictornaries

In [ ]:
# AI tag - #
timit_to_arpabet = {
    "aa": "aa",
    "ae": "ae",
    "ah": "ah",
    "ao": "ao", 
    "aw": "aw",
    "ax": "ax",  "ax-h": "ax",
    "axr": "axr",
    "ay": "ay", 
    "eh": "eh",
    "er": "er",
    "ey": "ey",
    "ih": "ih", 
    "ix": "ix",
    "iy": "iy",
    "ow": "ow",
    "oy": "oy", 
    "uh": "uh",
    "uw": "uw",
    "ux": "ux",
    "b": "b",   "bcl": "b",
    "ch": "ch",
    "d": "d",   "dcl": "d",
    "dh": "dh",
    "dx": "dx",
    "el": "el",
    "em": "em",
    "en": "en",
    "f": "f",
    "g": "g",   "gcl": "g",
    "hh": "hh", "hv": "hh",
    "jh": "jh",
    "k": "k",   "kcl": "k",
    "l": "l",
    "m": "m",
    "n": "n",
    "ng": "ng", "eng": "ng",
    "nx": "nx",
    "p": "p",   "pcl": "p",
    "q": "q",
    "r": "r",
    "s": "s",
    "sh": "sh",
    "t": "t",   "tcl": "t",
    "th": "th",
    "v": "v",
    "w": "w",
    "wh": "wh",
    "y": "y",
    "z": "z",
    "zh": "zh",
    "h#": "sil",
    "#h": "sil",
    "pau": "sil",
    "epi": "sil",
}

csd_ipa_to_arpabet = {
    "a": "aa",
    "ae": "ae",
    "ao": "ah",
    "oo": "ao",
    "au": "aw",
    "eo": "ax",
    # "--": "axr",
    "ai": "ay",
    "e": "eh",
    "er": "er",
    "ei": "ey",
    "i": "ih",
    # "--": "ix",
    "ii": "iy",
    "ou": "ow",
    "oi": "oy",
    "u": "uh",
    "uu": "uw",
    # "--": "ux",
    "b": "b",
    "ch": "ch",
    "d": "d",
    "dh": "dh",
    # "--": "dx",
    # "--": "el",
    # "--": "em",
    # "--": "en",
    "f": "f",
    "g": "g",
    "h": "hh",
    "j": "jh",
    "k": "k",
    "l": "l",
    "m": "m",
    "n": "n",
    "ng": "ng",
    # "--": "nx",
    "p": "p",
    # "--": "q",
    "r": "r",
    "s": "s",
    "sh": "sh",
    "t": "t",
    "th": "th",
    "v": "v",
    "w": "w",
    # "--": "wh",
    "y": "y",
    "z": "z",
    "zh": "zh",
}

buckeye_to_arpabet = {
    "aa": "aa",   "aan": "aa",  "a": "aa", "an": "aa",  
    "ae": "ae",   "aen": "ae",
    "ah": "ah",   "ahn": "ah",
    "ao": "ao",   "aon": "ao",
    "aw": "aw",   "awn": "aw",
    "ax": "ax",   "axn": "ax",  "x": "ax",     
    "axr": "axr", "axrn": "axr",
    "ay": "ay",   "ayn": "ay",
    "eh": "eh",   "ehn": "eh",
    "er": "er",   "ern": "er",
    "ey": "ey",   "eyn": "ey", "e": "ey",     
    "ih": "ih",   "ihn": "ih",
    "ix": "ix",   "ixn": "ix", "id": "ix",    
    "iy": "iy",   "iyn": "iy", "i": "iy",     
    "ow": "ow",   "own": "ow", "o": "ow",  
    "oy": "oy",   "oyn": "oy",
    "uh": "uh",   "uhn": "uh",
    "uw": "uw",   "uwn": "uw", "u": "uw",     
    "ux": "ux",   "uxn": "ux",
    "b": "b",   
    "d": "d",   
    "g": "g",   
    "p": "p",   
    "t": "t",   
    "k": "k",   
    "q": "q",
    "tq": "q",
    "dx": "dx",
    "nx": "nx",
    "el": "el",
    "em": "em",
    "en": "en",
    "eng": "en",
    "ch": "ch",
    "dh": "dh",
    "f": "f",
    "hh": "hh", "hhn": "hh", "h": "hh",
    "jh": "jh",
    "s": "s",
    "sh": "sh",
    "th": "th",
    "v": "v",
    "z": "z",
    "zh": "zh",
    "l": "l",
    "r": "r",
    "w": "w",
    "wh": "wh",
    "y": "y", "j": "y",
    "m": "m",
    "n": "n", "no": "n",
    "ng": "ng",
    
    "sil": "sil",     
    "pau": "sil",  
    "idu": "sil",    
    "cough": "sil", 
    "gasp": "sil",
    "iver": "sil",
    "iver-laugh": "sil",
    "laugh": "sil", 
    "throat": "sil",
    "sneeze": "sil",
    "noise": "sil",
    "vocnoise": "sil",
    "unknown": "sil",
    "<exclude-name>": "sil",
    "<exclude>": "sil",
    "exclude": "sil",
    "{e_trans}": "sil",
    "{b_trans}": "sil",
}

arpabet_phonetics_to_idx = {
    "sil": 0,
    "aa": 1,
    "ae": 2,
    "ah": 3,
    "ao": 4,
    "aw": 5,
    "ax": 6,
    "axr": 7,
    "ay": 8,
    "eh": 9,
    "er": 10,
    "ey": 11,
    "ih": 12,
    "ix": 13,
    "iy": 14,
    "ow": 15,
    "oy": 16,
    "uh": 17,
    "uw": 18,
    "ux": 19,
    "b": 20,
    "ch": 21,
    "d": 22,
    "dh": 23,
    "dx": 24,
    "el": 25,
    "em": 26,
    "en": 27,
    "f": 28,
    "g": 29,
    "hh": 30,
    "jh": 31,
    "k": 32,
    "l": 33,
    "m": 34,
    "n": 35,
    "ng": 36,
    "nx": 37,
    "p": 38,
    "q": 39,
    "r": 40,
    "s": 41,
    "sh": 42,
    "t": 43,
    "th": 44,
    "v": 45,
    "w": 46,
    "wh": 47,
    "y": 48,
    "z": 49,
    "zh": 50,
}

idx_to_arpabet_phonetics = {
    0: "sil",
    1: "aa",
    2: "ae",
    3: "ah",
    4: "ao",
    5: "aw",
    6: "ax",
    7: "axr",
    8: "ay",
    9: "eh",
    10: "er",
    11: "ey",
    12: "ih",
    13: "ix",
    14: "iy",
    15: "ow",
    16: "oy",
    17: "uh",
    18: "uw",
    19: "ux",
    20: "b",
    21: "ch", # sibilance
    22: "d",
    23: "dh",
    24: "dx",
    25: "el",
    26: "em",
    27: "en",
    28: "f",  # sibilance
    29: "g",
    30: "hh",
    31: "jh",
    32: "k",  # sibilance
    33: "l",
    34: "m",
    35: "n",
    36: "ng",
    37: "nx",
    38: "p",
    39: "q",
    40: "r",
    41: "s",  # sibilance
    42: "sh", # sibilance
    43: "t",  # sibilance
    44: "th", # sibilance
    45: "v",
    46: "w",
    47: "wh",
    48: "y",
    49: "z",
    50: "zh", # sibilance
}

arpabet_idx_to_idx_two = {
    21: 1,
    28: 1,
    32: 1,
    41: 1,
    42: 1,
    43: 1,
    44: 1,
    50: 1
}

idx_two_to_arpabet_phonetics = {
    0: "Other",
    1: "Sib"
}

arpabet_phonetics_to_idx_two = {
    "other": 0,
    "sib": 1,
}

arpabet_idx_to_idx_three = {
    21: 1,
    28: 1,
    32: 1,
    41: 2,
    42: 2,
    43: 2,
    44: 2,
    50: 2
}

idx_three_to_arpabet_phonetics = {
    0: "Other",
    1: "Air_Sib",
    2: "Cut_Sib"
}

arpabet_phonetics_to_idx_three = {
    "other" : 0,
    "air_sib": 1,
    "cut_sib": 2
}

arpabet_idx_to_idx_five = {
    # Sil: 0 , Vowels: 1, Nasals: 2, Fricatives: 3, Stops: 4
    0: 0, # Sil
    1: 1, # Vowels
    2: 1, # Vowels
    3: 1, # Vowels
    4: 1, # Vowels
    5: 1, # Vowels
    6: 1, # Vowels
    7: 1, # Vowels
    8: 1, # Vowels
    9: 1, # Vowels
    10: 1, # Vowels
    11: 1, # Vowels
    12: 1, # Vowels
    13: 1, # Vowels
    14: 1, # Vowels
    15: 1, # Vowels
    16: 1, # Vowels
    17: 1, # Vowels
    18: 1, # Vowels
    19: 1, # Vowels
    20: 4, # Stops
    21: 3, # Fricatives   # sibilance 
    22: 4, # Stops
    23: 3, # Fricatives
    24: 2, # Nasals
    25: 1, # Vowels
    26: 2, # Nasals
    27: 2, # Nasals
    28: 3, # Fricatives   # sibilance 
    29: 4, # Stops
    30: 3, # Fricatives
    31: 3, # Fricatives
    32: 4, # Stops        # sibilance 
    33: 1, # Vowels
    34: 2, # Nasals
    35: 2, # Nasals
    36: 2, # Nasals
    37: 2, # Nasals
    38: 4, # Stops
    39: 3, # Fricatives
    40: 1, # Vowels
    41: 3, # Fricatives   # sibilance 
    42: 3, # Fricatives   # sibilance 
    43: 4, # Stops        # sibilance 
    44: 3, # Fricatives   # sibilance 
    45: 3, # Fricatives
    46: 1, # Vowels
    47: 1, # Vowels
    48: 1, # Vowels
    49: 3, # Fricatives
    50: 3, # Fricatives   # sibilance 
}

idx_five_to_arpabet_phonetics = {
    0: "Sil",
    1: "Vowels",
    2: "Nasals",
    3: "Fricatives",
    4: "Stops"
}

arpabet_idx_to_idx_nine = {
    21: 1,
    28: 2,
    32: 3,
    41: 4,
    42: 5,
    43: 6,
    44: 7,
    50: 8
}

idx_nine_to_arpabet_phonetics = {
    0: "Other",
    1: "ch", # sibilance
    2: "f",  # sibilance
    3: "k",  # sibilance
    4: "s",  # sibilance
    5: "sh", # sibilance
    6: "t",  # sibilance
    7: "th", # sibilance
    8: "zh" # sibilance
}

arpabet_phonetics_to_idx_nine = {
    "other": 0,
    "ch": 1, # sibilance
    "f": 2,  # sibilance
    "k": 3,  # sibilance
    "s": 4,  # sibilance
    "sh": 5, # sibilance
    "t": 6,  # sibilance
    "th": 7, # sibilance
    "zh": 8 # sibilance
}

arpabet_phonetics_to_idx = {
    "other": 0,
    "sil": 0,
    "aa": 1,
    "ae": 2,
    "ah": 3,
    "ao": 4,
    "aw": 5,
    "ax": 6,
    "axr": 7,
    "ay": 8,
    "eh": 9,
    "er": 10,
    "ey": 11,
    "ih": 12,
    "ix": 13,
    "iy": 14,
    "ow": 15,
    "oy": 16,
    "uh": 17,
    "uw": 18,
    "ux": 19,
    "b": 20,
    "ch": 21, # sibilance
    "d": 22,
    "dh": 23,
    "dx": 24,
    "el": 25,
    "em": 26,
    "en": 27,
    "f": 28,  # sibilance
    "g": 29,
    "hh": 30,
    "jh": 31,
    "k": 32,  # sibilance
    "l": 33,
    "m": 34,
    "n": 45,
    "ng": 36,
    "nx": 37,
    "p": 38,
    "q": 39,
    "r": 40,
    "s": 41,  # sibilance
    "sh": 42, # sibilance
    "t": 43,  # sibilance
    "th": 44, # sibilance
    "v": 45,
    "w": 46,
    "wh": 47,
    "y": 48,
    "z": 49,
    "zh": 50, # sibilance
}

## Model

In [ ]:
# Define the Model Architecture
# AI tag - #
# ---------------------------------------------------------
class KClassificationModel(nn.Module):
    def __init__(self, input_dim, hidden_dim_one, hidden_dim_two, hidden_dim_three, num_classes):
        '''
        # [INPUT] (512) 32ms
        #   |
        # [Fully Con] (512, 512)
        #   |
        # [ReLU]
        #   |
        # [Fully Con] (512, 256)
        #   |
        # [ReLU]
        #   |
        # [Fully Con] (256, 51)
        #   |
        # [ReLU]
        #   |
        # [Fully Con] (51, 2)
        #   |
        # [OUTPUT log Softmax] (2)
        '''
        super(KClassificationModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim_one)
        self.fc2 = nn.Linear(hidden_dim_one, hidden_dim_two)
        self.fc3 = nn.Linear(hidden_dim_two, hidden_dim_three)
        self.fc4 = nn.Linear(hidden_dim_three, num_classes)
        self.act = nn.ReLU()
        
        self.softmax = nn.LogSoftmax(dim=-1)
        
    def forward(self, x):
        out = self.act(self.fc1(x))
        out = self.act(self.fc2(out))
        out = self.act(self.fc3(out))
        probs = self.softmax(self.fc4(out))
        return probs

## Data Loaders

In [ ]:
# Augmentaion Function
# AI tag - +
@torch.no_grad()
def apply_effects(_audio, sr):
    audio = torch.from_numpy(_audio).clone()

    # Phase Invert
    if random.random() < 0.4:
        audio = audio * -1
        
    # High Pass
    if random.random() < 0.4:
        low_cut = random.uniform(0, 150)
        audio = audio_F.highpass_biquad(audio, sr, low_cut)

    # Low Pass
    if random.random() < 0.4:
        high_cut = random.uniform(3000, 18000)
        audio = audio_F.lowpass_biquad(audio, sr, high_cut)

    # Reverb (Simple Delay/Echo)
    if random.random() < 0.4:
        reverb_mix = random.uniform(0.0, 0.5) 
        delay_ms = random.uniform(0.01, 4)
        delay_samples = int(delay_ms * sr)
        
        reverb_signal = torch.zeros_like(audio)
        if delay_samples < audio.shape[0]: # Guard against extreme small lengths
            reverb_signal[delay_samples:] = audio[:-delay_samples] * 0.6
            
        audio = (1 - reverb_mix) * audio + reverb_mix * reverb_signal
            
    return audio

In [ ]:
# AI tag - #
class MultiDataset(Dataset):
    def __init__(self, root_dir: str = "", dataset_name: str = "", augment: bool = False, samples = None, silibance: int = 2):
        """
        Args:
            root_dir (str): Root file directory for this specific dataset.
            dataset_name (str): The dataset footprint identifier used to manage target sample rates.
            augment (bool): Applies additional effects to the audio for better training.
            samples: A premade set of addresses.
            silibance (bool): Maps silibance instead of phonetics
        """
        self.root_dir = root_dir
        self.dataset_name = dataset_name
        self.augment = augment
        self.target_sr = self._lookup_sample_rate(self.dataset_name)
        self.silibance = silibance

        if samples is not None:
            self.samples = samples
        else:
            self.samples = []
            match self.dataset_name:
                case "TIMIT":
                    self._build_timit_dataset()
                case "Buckeye":
                    self._build_buckeye_dataset()
                case "NUS_48":
                    self._build_nus_dataset()
                case "CSD":
                    self._build_csd_dataset()
                case _:
                    raise FileNotFoundError(f"Not a valid dataset {self.dataset_name}")

    def _lookup_sample_rate(self, name):
        """Sample Rate Lookup."""
        sr_config_map = {
            "TIMIT": 16000,
            "Buckeye": 16000,
            "NUS_48": 16000,
            "CSD": 44100
        }
        return sr_config_map.get(name, 16000)

    def _build_timit_dataset(self):
        """
        Locates the TIMIT audio files and maps them directly to their companion 
        phonetic transcript locations (.PHN).
        """
        search_path = os.path.join(self.root_dir, "*", "*", "*.WAV")
        wav_files = glob.glob(search_path)
        
        for audio_path in wav_files:
            base_path, _ = os.path.splitext(audio_path)
            phn_path = base_path + ".PHN"
            
            if os.path.exists(phn_path):
                self.samples.append({
                    "dataset_name": self.dataset_name,
                    "audio_location": audio_path,
                    "text_location": phn_path
                    })

    def _build_nus_dataset(self):
        """
        Locates the NUS-48 audio files and maps them directly to their companion 
        phonetic transcript locations (.txt).
        """
        singer_dirs = glob.glob(os.path.join(self.root_dir, "????"))
        
        for singer_dir in singer_dirs:
            for sub_folder in ['read', 'sing']:
                target_dir = os.path.join(singer_dir, sub_folder)
                
                if os.path.exists(target_dir):
                    wav_files = glob.glob(os.path.join(target_dir, "*.wav"))
                    
                    for audio_path in wav_files:
                        base_path, _ = os.path.splitext(audio_path)
                        text_path = base_path + ".txt" 
                        
                        if os.path.exists(text_path):
                            self.samples.append({
                                "dataset_name": self.dataset_name,
                                "audio_location": audio_path,
                                "text_location": text_path
                            })

    def _build_buckeye_dataset(self):
        """
        Locates the Buckeye audio files and maps them directly to their companion 
        phonetic transcript locations (.phones).
        """
        search_path = os.path.join(self.root_dir, "*", "*", "*.wav")
        wav_files = glob.glob(search_path)
        
        for audio_path in wav_files:
            base_path, _ = os.path.splitext(audio_path)
            phones_path = base_path + ".phones"
            
            if os.path.exists(phones_path):
                self.samples.append({
                    "dataset_name": self.dataset_name,
                    "audio_location": audio_path,
                    "text_location": phones_path
                })
                
    def _build_csd_dataset(self):
        """
        Locates the CSD audio files and maps them directly to their companion 
        phonetic transcript locations (.csv).
        """
        root = Path(self.root_dir)
        
        for lang in ['english']: # 'korean'
            lang_dir = root / lang
            wav_dir = lang_dir / 'wav'
            csv_dir = lang_dir / 'csv'
            
            if not (wav_dir.exists() and csv_dir.exists()):
                continue
                
            for audio_path in wav_dir.glob("*.wav"):
                csv_path = csv_dir / f"{audio_path.stem}.csv"
                
                if csv_path.exists():
                    self.samples.append({
                        "dataset_name": self.dataset_name,
                        "audio_location": str(audio_path),
                        "text_location": str(csv_path)
                    })

    def __len__(self):
        return len(self.samples)

    def _parse_phonetic_phn(self, text_path):
        """
        Parses TIMIT .PHN sample boundaries directly at sample rate.
        Output Format: np.array([[start_sample, end_sample, phonetic], ...])
        """
        phonetic_intervals = []
        
        with open(text_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 3:
                    try:
                        start_sample = int(parts[0])
                        end_sample = int(parts[1])
                        
                        if end_sample - start_sample > 0:

                            arpabet_phone = timit_to_arpabet.get(parts[2].lower())
                            phonetic_symbol = arpabet_phonetics_to_idx.get(arpabet_phone, -1)
                            
                            phonetic_intervals.append([start_sample, end_sample, phonetic_symbol])
                    except ValueError:
                        continue
                        
        return np.array(phonetic_intervals)

    def _parse_phonetic_pho(self, text_path):
        """
        Parses Buckeye .phones files, skips the Xwaves header, 
        and converts seconds timestamps directly to sample indices.
        Output Format: np.array([[start_sample, end_sample, phonetic], ...])
        """
        phonetic_intervals = []
        
        with open(text_path, 'r', encoding='utf-8') as f:
            content = f.read()

        if '#' in content:
            data_section = content.split('#', 1)[1]
        else:
            data_section = content
            
        start_time = 0.0
        
        for line in data_section.strip().split('\n'):
            parts = line.strip().split()
            
            if len(parts) >= 3:
                try:
                    end_time = float(parts[0])
                    raw_phone = parts[2].split(';')[0]
                    phone_label = raw_phone.strip().lower()
                    
                    if end_time - start_time > 0:
                        _phone_label = phone_label.split('+')[0]
                        arpabet_phone = buckeye_to_arpabet.get(_phone_label, _phone_label)
                        phonetic_symbol = arpabet_phonetics_to_idx.get(arpabet_phone, arpabet_phone)

                        if phonetic_symbol == arpabet_phone:
                            print(text_path)
                        
                        start_sample = int(start_time * self.target_sr)
                        end_sample = int(end_time * self.target_sr)
                        
                        phonetic_intervals.append([start_sample, end_sample, phonetic_symbol])
                    
                    start_time = end_time
                    
                except ValueError:
                    continue
                    
        return np.array(phonetic_intervals)

    def _parse_phonetic_txt(self, text_path):
        """
        Parses text boundaries and calculates absolute audio sample lengths.
        Output Format: np.array([[start_sample, end_sample, phonetic], ...])
        """
        phonetic_intervals = []
        
        with open(text_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 3:
                    try:
                        start_time = float(parts[0])
                        end_time = float(parts[1])
                        if float(end_time) - float(start_time) > 0:
                            if parts[2].lower() == "sp":
                                parts[2] = "sil"
                            phonetic_symbol = arpabet_phonetics_to_idx.get(parts[2].lower(), -1)
                        
                        start_sample = int(start_time * self.target_sr)
                        end_sample = int(end_time * self.target_sr)
                        
                        phonetic_intervals.append([start_sample, end_sample, phonetic_symbol])
                    except ValueError:
                        continue
                        
        return np.array(phonetic_intervals, dtype=np.int64)

    def _parse_phonetic_csv(self, csv_path):
        """
        Parses CSV syllable boundaries, splits syllables into phonetics, 
        and calculates absolute audio sample lengths evenly distributed per phonetic.
        
        Expected CSV Columns: start, end, pitch, syllable
        Output Format: np.array([[start_sample, end_sample, phonetic_idx], ...])
        """
        phonetic_intervals = []
        
        with open(csv_path, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f, delimiter=',')
            
            for row in reader:
                try:
                    start_time = float(row['start'])
                    end_time = float(row['end'])
                    total_duration = end_time - start_time
                    
                    if total_duration <= 0:
                        continue
                    
                    syllable = row['syllable'].strip()
                    if not syllable:
                        continue
                        
                    phonetics = syllable.split('_')
                    num_phonetics = len(phonetics)
                    duration_per_phonetic = total_duration / num_phonetics
                    
                    for i, phone in enumerate(phonetics):
                        phone_start_time = start_time + (i * duration_per_phonetic)
                        phone_end_time = phone_start_time + duration_per_phonetic
                        
                        start_sample = int(phone_start_time * self.target_sr)
                        end_sample = int(phone_end_time * self.target_sr)
                        
                        arpabet_phone = csd_ipa_to_arpabet.get(phone.lower())
                        phonetic_symbol = arpabet_phonetics_to_idx.get(arpabet_phone, -1)
                        
                        phonetic_intervals.append([start_sample, end_sample, phonetic_symbol])
                        
                except (ValueError, KeyError):
                    continue
                    
        return np.array(phonetic_intervals, dtype=np.int64)

    def _create_alignment_target(self, audio_length: int, phonetic_intervals):
        """
        Creates a 1D tensor where every single audio sample index 
        is mapped to its corresponding phonetic ID.
        """
        target = torch.zeros(audio_length, dtype=torch.long)
        
        for row in phonetic_intervals:
            start_sample = int(row[0])
            end_sample = int(row[1])
            phonetic_id = int(row[2])

            if self.silibance == 2:
                target[start_sample:end_sample] = arpabet_idx_to_idx_two.get(phonetic_id, 0)
            elif self.silibance == 3:
                target[start_sample:end_sample] = arpabet_idx_to_idx_three.get(phonetic_id, 0)
            elif self.silibance == 5:
                target[start_sample:end_sample] = arpabet_idx_to_idx_five.get(phonetic_id, 0)
            elif self.silibance == 9:
                target[start_sample:end_sample] = arpabet_idx_to_idx_nine.get(phonetic_id, 0)
            else:
                target[start_sample:end_sample] = phonetic_id
            
        return target

    def __getitem__(self, idx):
        """
        Executes dynamic runtime translation to prevent memory leakages.
        Returns:
            audio_array (np.ndarray): Waveform trace managed by Librosa
            phonetic_array (np.ndarray): Mapped tracking boundaries [start, end, phonetic]
        """
        sample_info = self.samples[idx]
        
        audio_array, _ = librosa.load(
            sample_info["audio_location"], 
            sr=self.target_sr, 
            mono=True
        )

        if self.augment:
            audio_array = apply_effects(audio_array, self.target_sr)

        match sample_info["dataset_name"]:
            case "TIMIT":
                phonetic_array = self._create_alignment_target(len(audio_array), self._parse_phonetic_phn(sample_info["text_location"]))
            case "Buckeye":
                phonetic_array = self._create_alignment_target(len(audio_array), self._parse_phonetic_pho(sample_info["text_location"]))
            case "NUS_48":
                phonetic_array = self._create_alignment_target(len(audio_array), self._parse_phonetic_txt(sample_info["text_location"]))
            case "CSD":
                phonetic_array = self._create_alignment_target(len(audio_array), self._parse_phonetic_csv(sample_info["text_location"]))
            case _:
                raise FileNotFoundError(f"Not a valid dataset {self.dataset_name}")
                
        return audio_array, phonetic_array

## Set up

| Sample Rate |            1 |   128 |   256 |   512 |  1024 |
|------------:|-------------:|------:|------:|------:|------:|
|       16000 |    0.0000625 | 0.008 | 0.016 | 0.032 | 0.064 |
|       22050 | 0.0000453515 | 0.006 | 0.012 | 0.023 | 0.046 |
|       44100 | 0.0000226757 | 0.003 | 0.006 | 0.012 | 0.023 |
|       44800 | 0.0000208333 | 0.003 | 0.005 | 0.011 | 0.021 |
|-------------|--------------|-------|-------|-------|-------|
|       16000 |       0.0625 |     8 |    16 |    32 |    64 |
|       22050 |    0.0453515 |     6 |    12 |    23 |    46 |
|       44100 |    0.0226757 |     3 |     6 |    12 |    23 |
|       44800 |    0.0208333 |     3 |     5 |    11 |    21 |

In [ ]:
# Set Up Data
# AI tag - +
INPUT_FEATURES = 512  # Number of input features
HIDDEN_DIM_ONE = 512  # Hidden layer size
HIDDEN_DIM_TWO = 256  #
HIDDEN_DIM_THREE = 51 #
NUM_CLASSES = 2      # Total Classes [51 for phone and 2 for sibilance]
BATCH_SIZE = 1        # Only one batch as we don't have a collate function
HOP_SIZE = 128        # 128
SUB_BATCH_SIZE = 512  # The batches for windows
lr = 5e-4
epochs = 0

names = ["TIMIT", "NUS-48", "Buckeye", "CSD"]
Datasets = [name for i, name in enumerate(names) if DATASETS_TO_USE[i]]

slb = NUM_CLASSES

if verbose:
    print(f"Input Features: {INPUT_FEATURES}\nHidden Units: {HIDDEN_DIM_ONE}\nHidden Units (2nd layer): {HIDDEN_DIM_TWO}\nHidden Units (3rd Layer): {HIDDEN_DIM_THREE}\nNumber of Classes: {NUM_CLASSES}\nBatch Size: {BATCH_SIZE}\nHop Size: {HOP_SIZE}\nSub Batch Size: {SUB_BATCH_SIZE}\nLearning Rate: {lr}\nTotal Epochs: {epochs}\nUsing Datsets: {Datasets}")

In [ ]:
# Loading Model
# AI tag - #
torch.cuda.empty_cache()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = KClassificationModel(INPUT_FEATURES, HIDDEN_DIM_ONE, HIDDEN_DIM_TWO, HIDDEN_DIM_THREE, NUM_CLASSES).to(device)
criterion = nn.KLDivLoss(reduction="batchmean").to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)

if verbose:
    print(f"Using -> {device}")
    print(f"{model}\n")
    print(f"{criterion}\n")
    print(f"{optimizer}\n")

In [ ]:
# Loading Datasets
# AI tag - +
all_samples = []

if DATASETS_TO_USE[0]:
    timit_train_set = MultiDataset(
        root_dir="Dataset/TIMIT/TIMIT/TRAIN/",
        dataset_name="TIMIT",
        silibance = slb
    )
    
    timit_val_set = MultiDataset(
        root_dir="Dataset/TIMIT/TIMIT/TEST/",
        dataset_name="TIMIT",
        silibance = slb
    )
    if verbose:
        print(f"Loaded: TIMIT with {len(timit_train_set.samples + timit_val_set.samples)} samples")

if DATASETS_TO_USE[1]:
    nus_48_dataset = MultiDataset(
        root_dir="Dataset/NUS_48/",
        dataset_name="NUS_48",
        silibance = slb
    )
    all_samples += nus_48_dataset.samples
    if verbose:
        print(f"Loaded: NUS-48 with {len(nus_48_dataset.samples)} samples")

if DATASETS_TO_USE[2]:
    buckeye_dataset = MultiDataset(
        root_dir="Dataset/Buckeye/",
        dataset_name="Buckeye",
        silibance = slb
    )
    all_samples += buckeye_dataset.samples
    if verbose:
        print(f"Loaded: Buckeye with {len(buckeye_dataset.samples)} samples")

if DATASETS_TO_USE[3]:
    csd_dataset = MultiDataset(
        root_dir="Dataset/CSD/",
        dataset_name="CSD",
        silibance = slb
    )
    all_samples += csd_dataset.samples
    if verbose:
        print(f"Loaded: CSD with {len(csd_dataset.samples)} samples")

random.seed(42)
random.shuffle(all_samples)

split_idx = int(0.8 * len(all_samples))
train_files = all_samples[:split_idx]
val_files = all_samples[split_idx:]

if DATASETS_TO_USE[0]:
    train_files += timit_train_set.samples
    val_files += timit_val_set.samples

train_set = MultiDataset(
    samples = train_files,
    augment = Aug,
    silibance = slb
)

val_set = MultiDataset(
    samples = val_files,
    augment = False,
    silibance = slb
)

if verbose:
    print(f"Train set: {len(train_set)}, Val set: {len(val_set)}")

In [ ]:
# DATA LOADERS
# AI tag - #
train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE, 
    shuffle=True
)

val_loader = DataLoader(
    val_set, 
    batch_size=BATCH_SIZE, 
    shuffle=False
)

## Eval Functions

In [ ]:
# AI tag - *
def evaluate_time_weighted_metrics_multi(txt_path, target_labels=["Air_Sib", "Cut_Sib"], verbose=True):
    """
    Parses an overlapping prediction file, resolves time interval boundaries into 
    non-overlapping slices, calculates continuous time-weighted Precision, 
    Recall, and F1-Score for MULTIPLE target classes treated as positive events.
    Appends the report to the bottom of the input file.
    """
    # Standardize all target labels to lowercase for case-insensitive matching
    targets = {label.strip().lower() for label in target_labels} if isinstance(target_labels, (list, tuple, set)) else {target_labels.strip().lower()}
    
    time_points = set()
    segments = []

    # 1. Parse intervals and collect unique boundary timestamps
    with open(txt_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines:
            # Skip reading lines if the file was evaluated previously
            if line.startswith("=" * 10) or "TIME-WEIGHTED EVALUATION REPORT" in line:
                break

            parts = line.strip().split('\t')
            if len(parts) >= 4:
                try:
                    start_t = float(parts[0])
                    end_t = float(parts[1])
                    pred = parts[2].strip().lower()
                    truth = parts[3].strip().lower()

                    if end_t > start_t:
                        # Check if prediction or truth falls into ANY of the target classes
                        is_pred_target = pred in targets
                        is_truth_target = truth in targets

                        segments.append((start_t, end_t, is_pred_target, is_truth_target))
                        time_points.add(start_t)
                        time_points.add(end_t)
                except ValueError:
                    continue

    if not segments:
        if verbose:
            print("No valid data found in file.")
        return 0.0, 0.0, 0.0

    # 2. Slice time linearly into contiguous, non-overlapping atomic blocks
    sorted_times = sorted(list(time_points))
    
    tp_dur = 0.0  # True Positive duration
    fp_dur = 0.0  # False Positive duration
    fn_dur = 0.0  # False Negative duration
    tn_dur = 0.0  # True Negative duration

    for i in range(len(sorted_times) - 1):
        t_start = sorted_times[i]
        t_end = sorted_times[i + 1]
        duration = t_end - t_start

        if duration <= 1e-7:
            continue

        active_preds = [seg[2] for seg in segments if seg[0] <= t_start and seg[1] >= t_end]
        active_truths = [seg[3] for seg in segments if seg[0] <= t_start and seg[1] >= t_end]

        if not active_preds or not active_truths:
            continue

        is_pred_target = any(active_preds)
        is_truth_target = any(active_truths)

        if is_pred_target and is_truth_target:
            tp_dur += duration
        elif is_pred_target and not is_truth_target:
            fp_dur += duration
        elif not is_pred_target and is_truth_target:
            fn_dur += duration
        else:
            tn_dur += duration

    # 3. Compute continuous time-weighted metrics
    precision = tp_dur / (tp_dur + fp_dur) if (tp_dur + fp_dur) > 0 else 0.0
    recall = tp_dur / (tp_dur + fn_dur) if (tp_dur + fn_dur) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    # 4. Construct report block string
    target_str = ", ".join(list(targets))
    report_lines = [
        "\n",
        "=" * 60,
        f"TIME-WEIGHTED EVALUATION REPORT: Targets = [{target_str}]",
        "=" * 60,
        f"True Positive Time  (TP)    : {tp_dur:.4f} seconds",
        f"False Positive Time (FP)    : {fp_dur:.4f} seconds (Over-prediction)",
        f"False Negative Time (FN)    : {fn_dur:.4f} seconds (Missed target)",
        f"True Negative Time  (TN)    : {tn_dur:.4f} seconds",
        "-" * 60,
        f"PRECISION                   : {precision * 100:.2f}%",
        f"RECALL                      : {recall * 100:.2f}%",
        f"F1-SCORE                    : {f1 * 100:.2f}%",
        "=" * 60,
        "\n"
    ]
    report_str = "\n".join(report_lines)

    # 5. Append report to the text file
    with open(txt_path, 'a', encoding='utf-8') as f:
        f.write(report_str)

    if verbose:
        print(report_str)

    return precision, recall, f1

In [ ]:
# AI tag - #
def compress_segments(sequence, sample_rate=None, is_chunk_list=False):
    """
    Consolidated helper to compress sample-level IDs or chunk-level dictionaries 
    into contiguous timed phoneme blocks.
    """
    compressed = []
    
    # Setup initial segment properties based on the input type
    if is_chunk_list:
        curr_phone = sequence[0]["phoneme"]
        start_t = sequence[0]["start"]
        end_t = sequence[0]["end"]
    else:
        # Handling raw sample-level array/tensor
        sample_dur = 1.0 / sample_rate
        curr_phone = idx_to_arpabet_phonetics.get(int(sequence[0]), "err")
        start_t = 0.0

    for i, item in enumerate(sequence[1:], start=1):
        if is_chunk_list:
            next_phone = item["phoneme"]
            if next_phone != curr_phone:
                compressed.append({"start": start_t, "end": end_t, "phoneme": curr_phone})
                start_t, curr_phone = item["start"], next_phone
            end_t = item["end"]
        else:
            next_phone = idx_to_arpabet_phonetics.get(int(item), "err")
            if next_phone != curr_phone:
                compressed.append({"start": start_t, "end": i * sample_dur, "phoneme": curr_phone})
                start_t = i * sample_dur
                curr_phone = next_phone
                
    # Append the final item
    if is_chunk_list:
        compressed.append({"start": start_t, "end": end_t, "phoneme": curr_phone})
    else:
        compressed.append({"start": start_t, "end": len(sequence) * sample_dur, "phoneme": curr_phone})

    return compressed


def compress_combined_segments(predictions):
    """
    Compresses a sequential list of dictionaries containing both 
    predicted and truth phonemes into timed, contiguous blocks.
    A new block triggers if EITHER the prediction or the truth changes.
    """

    if not predictions:
        return []
    compressed = []

    # Initialize our tracking with the first element
    current_segment = {
        "start": predictions[0]["start"],
        "end": predictions[0]["end"],
        "phoneme": predictions[0]["phoneme"],
        "truth": predictions[0]["truth"]
    }

    for next_chunk in predictions[1:]:
        # Trigger a split if either the prediction or the ground truth shifts
        if (next_chunk["phoneme"] == current_segment["phoneme"] and 
            next_chunk["truth"] == current_segment["truth"]):

            # Extend the boundary of the current homogeneous block
            current_segment["end"] = next_chunk["end"]
        else:
            # Save the completed block and clone the new state
            compressed.append(current_segment)
            current_segment = {
                "start": next_chunk["start"],
                "end": next_chunk["end"],
                "phoneme": next_chunk["phoneme"],
                "truth": next_chunk["truth"]
            }

            
    # Append the final lingering chunk
    compressed.append(current_segment)
    return compressed 

def create_alignment_target(audio_length: int, phonetic_intervals):
    """
    Creates a 1D tensor where every single audio sample index 
    is mapped to its corresponding phonetic ID.
    """
    target = torch.zeros(audio_length, dtype=torch.long)
    
    for row in phonetic_intervals:
        start_sample = int(row[0])
        end_sample = int(row[1])
        phonetic_id = int(row[2])

        target[start_sample:end_sample] = phonetic_id
        
    return target

def parse_phonetic_txt(text_path, sr, classes):
    """
    Parses text boundaries and calculates absolute audio sample lengths.
    Output Format: np.array([[start_sample, end_sample, phonetic], ...])
    """
    phonetic_intervals = []
    
    with open(text_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 3:
                try:
                    start_time = float(parts[0])
                    end_time = float(parts[1])
                    phonetic_symbol = -2
                    if float(end_time) - float(start_time) > 0:
                        if parts[2].lower() == "sp":
                            parts[2] = "sil"
                        if classes == 2:
                            phonetic_symbol = arpabet_phonetics_to_idx_two.get(parts[2].lower(), -1)
                        elif classes == 3:
                            phonetic_symbol = arpabet_phonetics_to_idx_three.get(parts[2].lower(), -1)
                        elif classes == 9:
                            phonetic_symbol = arpabet_phonetics_to_idx_nine.get(parts[2].lower(), -1)
                        elif classes == 51:
                            phonetic_symbol = arpabet_phonetics_to_idx.get(parts[2].lower(), -1)
                    
                    start_sample = int(start_time * sr)
                    end_sample = int(end_time * sr)
                    
                    phonetic_intervals.append([start_sample, end_sample, phonetic_symbol])
                except ValueError:
                    continue
                    
    return np.array(phonetic_intervals, dtype=np.int64)

In [ ]:
# AI tag - #
def evaluation_loop(loader, NUM_CLASSES, model, csd="", sr: int = 16000, batch_size: int = 512):
    p_score, r_score, f1_score = 0.0, 0.0, 0.0
    count = 0
    name = ""
    
    test_classes = 9 if NUM_CLASSES == 51 else NUM_CLASSES
    
    with torch.no_grad():
        for raw_inputs, raw_labels in tqdm(loader, desc="Evaluation", leave=False):
            sr = int(raw_inputs[6:11])
            name = f"{raw_inputs[6:16]}_{NUM_CLASSES}"
            if csd != "":
                name = f"{name}_{csd}"
            
            audio_array, _ = librosa.load(
                raw_inputs, 
                sr=sr, 
                mono=True
            )
            audio = torch.from_numpy(audio_array)

            phonetic_array = create_alignment_target(len(audio), parse_phonetic_txt(raw_labels, sr, NUM_CLASSES))
            
            predictions = []
            
            full_targets = phonetic_array # Shape [total_samples,]    
                
            chunk_cutoff = len(full_targets) % INPUT_FEATURES
            if chunk_cutoff != 0:
                padding_needed = INPUT_FEATURES - chunk_cutoff
                audio = F.pad(audio, (0, padding_needed), mode='constant', value=0.0)
                full_targets = F.pad(full_targets, (0, padding_needed), mode='constant', value=0)

            
            audio_chunks = audio.unfold(0, INPUT_FEATURES, HOP_SIZE).to(device)
            num_chunks = audio_chunks.size(0)
            
            means = audio_chunks.mean(dim=1, keepdim=True)
            stds = audio_chunks.std(dim=1, keepdim=True)
            audio_chunks = torch.where(stds > 1e-4, (audio_chunks - means) / stds, torch.zeros_like(audio_chunks))
            
            for i in range(0, num_chunks, batch_size):
                # Slice the batch and add the expected channel dimension [B, 1, INPUT_FEATURES]
                X_batch = audio_chunks[i : i + batch_size].unsqueeze(1)
                
                logits = model(X_batch)
                
                if logits.dim() == 3 and logits.size(1) == 1:
                    logits = logits.squeeze(1)
                
                # Get predicted classes for the entire batch at once
                predicted_classes = torch.argmax(logits, dim=-1).cpu().tolist()
                
                for j, predicted_class in enumerate(predicted_classes):
                    chunk_idx = i + j
                    
                    start_sample = chunk_idx * HOP_SIZE
                    end_sample = start_sample + INPUT_FEATURES
                    
                    start_time_sec = start_sample / sr
                    end_time_sec = end_sample / sr

                    mid_sample = start_sample + (INPUT_FEATURES // 2)
                    # Handle edge case where mid_sample might exceed padded length
                    mid_sample = min(mid_sample, len(full_targets) - 1)
                    true_class = full_targets[mid_sample].item()

                    #print(f"{predicted_class}, {true_class}")
                    if NUM_CLASSES == 2:
                        phoneme_label = idx_two_to_arpabet_phonetics.get(predicted_class, f"err_{predicted_class}")
                        true_label = idx_two_to_arpabet_phonetics.get(true_class, f"err_{true_class}")
                    elif NUM_CLASSES == 3:
                        phoneme_label = idx_three_to_arpabet_phonetics.get(predicted_class, f"err_{predicted_class}")
                        true_label = idx_three_to_arpabet_phonetics.get(true_class, f"err_{true_class}")
                    elif NUM_CLASSES == 5:
                        phoneme_label = idx_five_to_arpabet_phonetics.get(predicted_class, f"err_{predicted_class}")
                        true_label = idx_five_to_arpabet_phonetics.get(true_class, f"err_{true_class}")
                    elif NUM_CLASSES == 9:
                        phoneme_label = idx_nine_to_arpabet_phonetics.get(predicted_class, f"err_{predicted_class}")
                        true_label = idx_nine_to_arpabet_phonetics.get(true_class, f"err_{true_class}")
                    else:
                        phoneme_label = idx_to_arpabet_phonetics.get(predicted_class, f"err_{predicted_class}")
                        true_label = idx_to_arpabet_phonetics.get(true_class, f"err_{true_class}")
                    #print(f"{phoneme_label}, {true_label}")   
                    predictions.append({
                        "start": start_time_sec,
                        "end": end_time_sec,
                        "phoneme": phoneme_label,
                        "truth": true_label
                    })

            predictions = compress_combined_segments(predictions)
            with open(f"Texts/predicted_{name}.txt", "w") as f:
                for seg in predictions:
                    f.write(f"{seg['start']:.4f}\t{seg['end']:.4f}\t{seg['phoneme']}\t{seg['truth']}\n")
            print(f"Saved predictions to 'predicted_{name}.txt'")

            if test_classes == 2:
                target_labels = "Sib"
            elif test_classes == 3:
                target_labels = ["Air_Sib", "Cut_Sib"]
            elif test_classes == 9:
                target_labels = ["ch", "f", "k","s","sh","t","th","zh"]
                
            p_s, r_s, f_s = evaluate_time_weighted_metrics_multi(f"Texts/predicted_{name}.txt", target_labels=target_labels)
            

## Training

In [ ]:
# AI tag - #
def training_loop(loader, batch_size: int, epoch: int, epochs: int, desc: str = "", training: bool = True):
    loop_loss = 0.0
    loop_loss_max = 0.0
    
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for raw_inputs, raw_labels in tqdm(loader, desc=f"Epoch {epoch+1}/{epochs} {desc}", leave=False):
            audio = raw_inputs.squeeze(0)        # Shape [total_samples,]
            full_targets = raw_labels.squeeze(0) # Shape [total_samples,]
            
            chunk_cutoff = len(full_targets) % INPUT_FEATURES
            if chunk_cutoff != 0:
                padding_needed = INPUT_FEATURES - chunk_cutoff
                audio = F.pad(audio, (0, padding_needed), mode='constant', value=0.0)
                full_targets = F.pad(full_targets, (0, padding_needed), mode='constant', value=0)
            
            audio_chunks = audio.unfold(0, INPUT_FEATURES, HOP_SIZE).to(device)
            target_windows = full_targets.unfold(0, INPUT_FEATURES, HOP_SIZE)
            num_chunks = audio_chunks.size(0)
            
            means = audio_chunks.mean(dim=1, keepdim=True)
            stds = audio_chunks.std(dim=1, keepdim=True)
            audio_chunks = torch.where(stds > 1e-4, (audio_chunks - means) / stds, torch.zeros_like(audio_chunks))
            
            num_windows, window_length = target_windows.shape
            counts = torch.zeros(num_windows, NUM_CLASSES, dtype=torch.float, device=target_windows.device)
            ones = torch.ones_like(target_windows, dtype=torch.float)
            counts.scatter_add_(1, target_windows.long(), ones)
            target_distributions = counts / (counts.sum(dim=1, keepdim=True) + 1e-8)
            target_distributions = target_distributions.to(device)
            
            file_loss = 0.0
            file_loss_max = 0.0
            
            for i in range(0, num_chunks, batch_size):
                X_batch = audio_chunks[i : i + batch_size].unsqueeze(1)
                y_batch = target_distributions[i : i + batch_size]
                
                logits = model(X_batch)
                if logits.dim() == 3 and logits.size(1) == 1:
                    logits = logits.squeeze(1)
                
                loss = criterion(logits, y_batch)
                
                if training:
                    optimizer.zero_grad()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                else:
                    win_slice = target_windows[i : i + batch_size]
                    dominant_ids = torch.mode(win_slice, dim=1).values
                    hard_targets = F.one_hot(dominant_ids.long(), num_classes=NUM_CLASSES).float().to(device)
                    
                    max_loss = criterion(logits, hard_targets)
                    file_loss_max += max_loss.item() * len(win_slice)
                    
                file_loss += loss.item() * len(X_batch)
                
            loop_loss += (file_loss / num_chunks)
            loop_loss_max += (file_loss_max / num_chunks)
                
    avg_loss = loop_loss / len(loader)
    avg_loop_loss_max = loop_loss_max / len(loader)
    return avg_loss, avg_loop_loss_max

In [ ]:
# Training Step
# ---------------------------------------------------------
# AI tag - +
# [1] - Training
# [2] - Evaluation (Loss)
# [3] - Saving

best_val_loss = 100.0

if use_pretrained:
    path = pretrained_model.split(".")
    splits = path[0].split("_")
    best_val_loss = float(str(splits[-2] + "." + splits[-1]))
    checkpoint = torch.load(pretrained_model)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)

if verbose:
    print(f"Current best loss: {best_val_loss}")

for epoch in tqdm(range(epochs), desc="Epoch Training", leave=False):
    # [1]
    model.train()  
    train_loss, _ = training_loop(train_loader, batch_size=SUB_BATCH_SIZE, desc="[Tra]", epoch=epoch, epochs=epochs)

    # [2]
    model.eval()
    val_loss, val_max_loss = training_loop(val_loader, batch_size=SUB_BATCH_SIZE, desc="[Val]", epoch=epoch, epochs=epochs, training=False)

    # [3]
    saving_text = ""
    if val_loss < best_val_loss:
        loss_str = f"{val_loss:.04f}".replace(".", "_")
        save_path = f"Models/{save_name}_{loss_str}.pth"
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': best_val_loss,
        }, save_path)
        saving_text = "| New Best Model Saved"
        
    print(f"Epoch {epoch+1:2}: Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Max Loss: {val_max_loss:.4f} {saving_text}")

## Model to JSON

In [ ]:
# Export model weights
# AI tag - #
if export:
    if use_pretrained:
        model = KClassificationModel(INPUT_FEATURES, HIDDEN_DIM_ONE, HIDDEN_DIM_TWO, HIDDEN_DIM_THREE, NUM_CLASSES).to(device)
        checkpoint = torch.load(pretrained_model)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)
        _path = pretrained_model.split(".")
        _path = _path[0]
        path = str(_path).split("/")
        save_name = path[1]
        
    class EncodeTensor(JSONEncoder,Dataset):
        def default(self, obj):
            if isinstance(obj, torch.Tensor):
                return obj.cpu().detach().numpy().tolist()
            return super(json.NpEncoder, self).default(obj)
    
    with open(f"JSON/{save_name}_model.json", 'w') as json_file:
      json.dump(model.state_dict(), json_file,cls=EncodeTensor)
        
    if verbose:
        print(f"Model saved to .JSON with name 'JSON/{save_name}_model.json'")

## Using the Model

In [ ]:
# AI tag - +
def run_eval(model_path, classes, csd):
    INPUT_FEATURES = 512  # Number of input features
    HIDDEN_DIM_ONE = 512  # Hidden layer size
    HIDDEN_DIM_TWO = 256  #
    HIDDEN_DIM_THREE = 51 #
    NUM_CLASSES = classes # Total Classes [51 for phone and 2 for sibilance]
    BATCH_SIZE = 1        # Only one batch as we don't have a collate function
    HOP_SIZE = 128        # 128
    SUB_BATCH_SIZE = 512
    
    audio_files = [
        ("Audio/16000_spoken.wav", f"Audio/spoken_{NUM_CLASSES}.txt"),
        ("Audio/16000_sung.wav", f"Audio/sung_{NUM_CLASSES}.txt"),
        ("Audio/44100_spoken.wav", f"Audio/spoken_{NUM_CLASSES}.txt"),
        ("Audio/44100_sung.wav", f"Audio/sung_{NUM_CLASSES}.txt")
    ]

    torch.cuda.empty_cache()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using -> {device}")
    
    model = KClassificationModel(INPUT_FEATURES, HIDDEN_DIM_ONE, HIDDEN_DIM_TWO, HIDDEN_DIM_THREE, NUM_CLASSES).to(device)
    
    checkpoint = torch.load(model_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    model.to(device)
    model.eval()
    print(f"Using Model {model_path}")
    
    with torch.no_grad():
        evaluation_loop(audio_files, NUM_CLASSES=NUM_CLASSES, model=model, csd=csd)
    

if eval_bool or True:
    # run_eval("Models/Full_no_aug_2c_e6_[ReLU]_0_2130.pth", classes=2) # 2 Classes
    # run_eval("Models/Full_no_aug_3c_e7_[ReLU]_0_2865.pth", classes=3) # 3 Classes
    # run_eval("Models/Full_no_aug_9c_e12_[ReLU]_0_3597.pth", classes=9) # 9 Classes
    # run_eval("Models/Full_aug_51c_e89_[ReLU]_1_7183.pth", classes=51) # 51 Classes
    # run_eval("Models/CSD_no_aug_2_16000_0_2851.pth", classes=2, csd="low") # 2 Classes
    run_eval("Models/CSD_no_aug_2_44100_0_2865.pth", classes=2, csd="high") # 2 Classes
